In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


# ====== Step 1: 准备工具 ======
# 工具
@tool
def get_weather(city: str) -> str:
    """获取城市天气"""
    weather_data = {
        "北京": "晴天 25度",
        "上海": "多云 28度",
        "杭州": "小雨 22度",
    }
    return weather_data.get(city, f"未找到{city}的天气信息")

# 组成工具数组
tools = [get_weather]

# ====== Step 2: 准备模型，绑定工具 ======
# 定义模型
model = init_chat_model(
    'deepseek-v4-flash',
    extra_body = {'thinking': {'type': 'disabled'}}
)
# 绑定工具到模型，与create_agent类似，将来LangChain会自动帮我们拼接tool信息发给模型
model_with_tools = model.bind_tools(tools)


# ====== Step 3: 定义Node ======
def llm_node(state: MessagesState):
    """LLM节点：调用模型，可以选择调用工具"""
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# ====== Step 4: 构建图 ======
builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")  # 工具执行后回到LLM，形成循环
agent_graph = builder.compile()

# ====== Step 5: 调用 ======
result = agent_graph.invoke({
    "messages": [HumanMessage(content="北京和杭州今天天气怎么样？")]
})
for m in result['messages']:
    m.pretty_print()

================================ Human Message =================================

北京和杭州今天天气怎么样？
================================== Ai Message ==================================

我来帮您查询北京和杭州今天的天气情况。
Tool Calls:
  get_weather (call_00_0cQWvger03OsJ4pkfqCW4214)
 Call ID: call_00_0cQWvger03OsJ4pkfqCW4214
  Args:
    city: 北京
  get_weather (call_01_WNawSzNB4RSaOwq9I9vs7172)
 Call ID: call_01_WNawSzNB4RSaOwq9I9vs7172
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

晴天 25度
================================= Tool Message =================================
Name: get_weather

小雨 22度
================================== Ai Message ==================================

您查询的两座城市今天天气如下：

| 城市 | 天气 | 气温 |
|------|------|------|
| 北京 | ☀️ 晴天 | 25°C |
| 杭州 | 🌧️ 小雨 | 22°C |

**小贴士**：北京今天晴朗舒适，适合外出活动；杭州有降雨，出门记得带上雨伞，注意防滑哦！
